# 機率與統計導論
在本筆記本中，我們將實作之前討論過的一些概念。許多機率與統計的概念都在 Python 的主要資料處理函式庫中有很好的支援，例如 `numpy` 和 `pandas`。


In [ ]:
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt

## 隨機變數與分佈
讓我們先從 0 到 9 的均勻分佈抽取 30 個樣本值。我們也會計算平均值與變異數。


In [ ]:
sample = [ random.randint(0,10) for _ in range(30) ]
print(f"Sample: {sample}")
print(f"Mean = {np.mean(sample)}")
print(f"Variance = {np.var(sample)}")

要直觀地估計樣本中有多少個不同的值，我們可以繪製 <strong>直方圖</strong>：


In [ ]:
plt.hist(sample)
plt.show()

## 分析實際數據

均值和變異數在分析真實世界數據時非常重要。讓我們從 [SOCR MLB Height/Weight Data](http://wiki.stat.ucla.edu/socr/index.php/SOCR_Data_MLB_HeightsWeights) 載入有關棒球運動員的數據


In [ ]:
df = pd.read_csv("../../data/SOCR_MLB.tsv",sep='\t', header=None, names=['Name','Team','Role','Weight','Height','Age'])
df


> 我們在此使用一個名為 [**Pandas**](https://pandas.pydata.org/) 的套件來進行資料分析。在本課程稍後會討論更多有關 Pandas 以及在 Python 中處理資料的方法。

讓我們計算年齡、身高和體重的平均值：


In [ ]:
df[['Age','Height','Weight']].mean()

現在讓我們專注於身高，計算標準差和變異數： 


In [ ]:
print(list(df['Height'])[:20])

In [ ]:
mean = df['Height'].mean()
var = df['Height'].var()
std = df['Height'].std()
print(f"Mean = {mean}\nVariance = {var}\nStandard Deviation = {std}")

除了平均值外，也有必要觀察中位數和四分位數。它們可以使用<strong>盒狀圖</strong>來視覺化：


In [ ]:
plt.figure(figsize=(10,2))
plt.boxplot(df['Height'].ffill(), orientation='horizontal', showmeans=True)
plt.grid(color='gray', linestyle='dotted')
plt.tight_layout()
plt.show()

我們亦可以繪製數據集子集的箱形圖，例如按球員角色分組。


In [ ]:
df.boxplot(column='Height', by='Role', figsize=(10,8))
plt.xticks(rotation='vertical')
plt.tight_layout()
plt.show()

> <strong>注意</strong>：此圖表顯示，平均而言，一壘手的身高比二壘手的身高高。稍後我們將學習如何更正式地檢驗這個假設，以及如何證明我們的數據在統計上具有顯著性以支持此結論。  

年齡、身高和體重都是連續隨機變數。你認為它們的分佈是什麼？一個好方法是繪製數值的直方圖： 


In [ ]:
df['Weight'].hist(bins=15, figsize=(10,6))
plt.suptitle('Weight distribution of MLB Players')
plt.xlabel('Weight')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## 常態分佈

讓我們建立一個人工的體重樣本，其分佈符合常態分佈，且具有與我們實際數據相同的平均值和變異數：


In [ ]:
generated = np.random.normal(mean, std, 1000)
generated[:20]

In [ ]:
plt.figure(figsize=(10,6))
plt.hist(generated, bins=15)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10,6))
plt.hist(np.random.normal(0,1,50000), bins=300)
plt.tight_layout()
plt.show()

由於現實生活中大多數數值都呈常態分佈，我們不應該使用均勻隨機數產生器來生成樣本數據。以下是如果我們嘗試用均勻分佈（由 `np.random.rand` 生成）來生成體重時會發生的情況：


In [ ]:
wrong_sample = np.random.rand(1000)*2*std+mean-std
plt.figure(figsize=(10,6))
plt.hist(wrong_sample)
plt.tight_layout()
plt.show()

## 信心區間

現在讓我們計算棒球運動員體重和身高的信心區間。我們將使用[這段stackoverflow討論](https://stackoverflow.com/questions/15033511/compute-a-confidence-interval-from-sample-data)中的程式碼：


In [ ]:
import scipy.stats

def mean_confidence_interval(data, confidence=0.95):
    a = 1.0 * np.array(data)
    n = len(a)
    m, se = np.mean(a), scipy.stats.sem(a)
    h = se * scipy.stats.t.ppf((1 + confidence) / 2., n-1)
    return m, h

for p in [0.85, 0.9, 0.95]:
    m, h = mean_confidence_interval(df['Weight'].ffill(),p)
    print(f"p={p:.2f}, mean = {m:.2f} ± {h:.2f}")

## 假設檢定

讓我們探索棒球選手數據集中不同的角色：


In [ ]:
df.groupby('Role').agg({ 'Weight' : 'mean', 'Height' : 'mean', 'Age' : 'count'}).rename(columns={ 'Age' : 'Count'})

讓我們測試「一壘手比二壘手高」的假設。最簡單的方法是測試信賴區間： 


In [ ]:
for p in [0.85,0.9,0.95]:
    m1, h1 = mean_confidence_interval(df.loc[df['Role']=='First_Baseman',['Height']],p)
    m2, h2 = mean_confidence_interval(df.loc[df['Role']=='Second_Baseman',['Height']],p)
    print(f'Conf={p:.2f}, 1st basemen height: {m1-h1[0]:.2f}..{m1+h1[0]:.2f}, 2nd basemen height: {m2-h2[0]:.2f}..{m2+h2[0]:.2f}')

我們可以看到這些區間是沒有重疊的。

一個更統計上正確證明假設的方法是使用 **Student t檢驗**：


In [ ]:
from scipy.stats import ttest_ind

tval, pval = ttest_ind(df.loc[df['Role']=='First_Baseman',['Height']], df.loc[df['Role']=='Second_Baseman',['Height']],equal_var=False)
print(f"T-value = {tval[0]:.2f}\nP-value: {pval[0]}")

`ttest_ind` 函數返回的兩個值是：
* p 值可以被視為兩個分布具有相同期望值的機率。在我們的例子中，p 值非常低，代表有強烈證據支持一壘手身高較高。
* t 值是用於 t 檢定的歸一化平均差的中介值，並且會與對應信心水準的閾值進行比較。


## 使用中心極限定理模擬常態分佈

Python 中的偽隨機生成器被設計為產生均勻分佈。如果我們想創建一個常態分佈的生成器，我們可以使用中心極限定理。要獲得一個常態分佈的值，我們只需計算一組均勻生成樣本的平均值。


In [ ]:
def normal_random(sample_size=100):
    sample = [random.uniform(0,1) for _ in range(sample_size) ]
    return sum(sample)/sample_size

sample = [normal_random() for _ in range(100)]
plt.figure(figsize=(10,6))
plt.hist(sample)
plt.tight_layout()
plt.show()

## 相關性與邪惡棒球公司

相關性讓我們能夠找到資料序列之間的關聯。在我們的玩具範例中，假設有一家邪惡的棒球公司，他們根據球員的身高支付薪水——球員越高，所得的錢越多。假設有基本薪資1000美元，並根據身高提供0到100美元的額外獎金。我們將採用MLB的真實球員，計算他們的想像薪水：


In [ ]:
heights = df['Height'].ffill()
salaries = 1000+(heights-heights.min())/(heights.max()-heights.mean())*100
print(list(zip(heights, salaries))[:10])

現在讓我們計算這些序列的共變異數與相關係數。`np.cov` 會給我們一個所謂的 <strong>共變異數矩陣</strong>，它是共變異數在多個變數上的擴展。共變異數矩陣 $M$ 的元素 $M_{ij}$ 是輸入變數 $X_i$ 與 $X_j$ 之間的共變異數，而對角線上的值 $M_{ii}$ 是 $X_i$ 的變異數。同理地，`np.corrcoef` 會給我們 <strong>相關係數矩陣</strong>。


In [ ]:
print(f"Covariance matrix:\n{np.cov(heights, salaries)}")
print(f"Covariance = {np.cov(heights, salaries)[0,1]}")
print(f"Correlation = {np.corrcoef(heights, salaries)[0,1]}")

相關係數等於1代表兩個變量之間存在強烈的<strong>線性關係</strong>。我們可以透過將一個變量對另一個變量作圖來直觀地看到線性關係：


In [ ]:
plt.figure(figsize=(10,6))
plt.scatter(heights,salaries)
plt.tight_layout()
plt.show()

讓我們看看當關係不是線性時會發生什麼。假設我們的公司決定隱藏身高和薪水之間明顯的線性依賴，並在公式中引入一些非線性，例如 `sin`：


In [ ]:
salaries = 1000+np.sin((heights-heights.min())/(heights.max()-heights.mean()))*100
print(f"Correlation = {np.corrcoef(heights, salaries)[0,1]}")

在這個例子中，相關性稍微小了一點，但仍然相當高。現在，為了讓這個關係變得不那麼明顯，我們可能想要通過加上一些隨機變量來增加一些額外的隨機性到薪水中。讓我們看看會發生什麼： 


In [ ]:
salaries = 1000+np.sin((heights-heights.min())/(heights.max()-heights.mean()))*100+np.random.random(size=len(heights))*20-10
print(f"Correlation = {np.corrcoef(heights, salaries)[0,1]}")

In [ ]:
plt.figure(figsize=(10,6))
plt.scatter(heights, salaries)
plt.tight_layout()
plt.show()

> 你能猜到為什麼這些點會排列成這樣的垂直線嗎？

我們已經觀察到像薪水這種人工設計的概念與觀察變量<em>身高</em>之間的相關性。現在讓我們也看看兩個觀察變量，例如身高和體重，是否也有相關性：


In [ ]:
np.corrcoef(df['Height'].ffill(),df['Weight'])

不幸地，我們沒有得到任何結果——只有一些奇怪的 `nan` 值。這是因為我們序列中的某些值是未定義的，表示為 `nan`，這導致運算結果也未定義。通過查看矩陣，我們可以看到 `Weight` 是出問題的欄位，因為已經計算了 `Height` 值之間的自相關。

> 這個例子展示了<strong>資料準備</strong>和<strong>清理</strong>的重要性。沒有適當的資料，我們無法計算任何東西。

讓我們使用 `fillna` 方法填補遺失值，並計算相關性：


In [ ]:
np.corrcoef(df['Height'].ffill(), df['Weight'])

確實存在相關性，但不像我們的人為範例中那麼強烈。事實上，如果我們查看一個值與另一個值的散點圖，這種關係會不那麼明顯： 


In [ ]:
plt.figure(figsize=(10,6))
plt.scatter(df['Weight'],df['Height'])
plt.xlabel('Weight')
plt.ylabel('Height')
plt.tight_layout()
plt.show()

## 總結

在此筆記本中，我們學會了如何對數據進行基本操作以計算統計函數。我們現在知道如何使用完整的數學和統計工具來驗證一些假設，以及如何根據數據樣本計算任意變量的置信區間。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件由 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻譯而成。雖然我們致力於確保準確性，但請注意，機器自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議進行專業人工翻譯。我們不對因使用本翻譯而產生的任何誤解或誤釋承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
